- **학습 목표**: 여러 모델을 같은 인터페이스로 다룰 수 있는 구조를 직접 설계하고, decorator로 공통 관심사(로깅·타이밍)를 분리할 수 있다.
- **핵심 개념**:
  - **추상 베이스 클래스(ABC)**: 하위 클래스가 반드시 구현해야 할 메서드를 강제합니다.
  - **decorator**: 함수를 감싸서 원래 로직을 건드리지 않고 부가 기능(시간 측정 등)을 추가합니다. `functools.wraps`를 쓰는 이유도 확인해보세요.

In [22]:
import functools
import time
from abc import ABC, abstractmethod

In [28]:
def log_timing(func):
  """
  요구사항:
  - 감싼 함수의 실행 시간을 측정해 "[함수이름] 0.0123초" 형태로 출력.
  - functools.wraps를 사용해 원래 함수의 __name__과 docstring을 보존할 것.
  """
  @functools.wraps(func)
  def wrapper(*arg,**kwargs):
    start = time.time()
    result = func(*arg,**kwargs)
    end = time.time()
    print(f"[{func.__name__}] {end-start:.4f}초")
    return result
  return wrapper

class BaseClassifierWrapper(ABC):
  """
  요구사항:
  - __init__(name): 이름과 fitted 상태를 저장.
  - _build_model(): @abstractmethod. 하위 클래스가 실제 sklearn 모델을 반환.
  - fit(X, y): @log_timing 적용. _build_model()로 모델을 만들고 학습, self 반환.
  - predict_proba(X): @log_timing 적용. fit되지 않았으면 명확한 에러.
                      양성 클래스 확률 1차원 배열 반환.
  - __repr__: 클래스명, name, fitted 상태가 보이게.
  """
  def __init__(self,name):
    self.name = name
    self.fitted = False
    self.model = None
  @abstractmethod
  def _build_model(self):
    pass
  @log_timing
  def fit(self, X, y):
    self.model = self._build_model()
    self.model.fit(X,y)
    self.fitted = True
    return self

  @log_timing
  def predict_proba(self, X):
    if not self.fitted:
      raise Exception("모델을 학습해주세요.")
    return self.model.predict_proba(X)[:,1]

  def __repr__(self):
    return f"{self.__class__.__name__}(name='{self.name}', fitted={self.fitted})"

# 위를 상속해서 LogisticWrapper(C=1.0)와 RandomForestWrapper(n_estimators=100)를
# 각각 구현할 것. 두 클래스 모두 _build_model()만 다르게 채우면 되도록 설계.
class LogisticWrapper(BaseClassifierWrapper):
  def __init__(self,C=1.0):
    super().__init__("logistic")
    self.C = C
  def _build_model(self):
    from sklearn.linear_model import LogisticRegression
    return LogisticRegression(C=self.C)

class RandomForestWrapper(BaseClassifierWrapper):
  def __init__(self,n_estimators=100):
    super().__init__("random_forest")
    self.n_estimators = n_estimators
  def _build_model(self):
    from sklearn.ensemble import RandomForestClassifier
    return RandomForestClassifier(n_estimators=self.n_estimators)

In [24]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import roc_auc_score

In [29]:
X, y = make_classification()
X_train, X_test, y_train, y_test = train_test_split(X,y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
log_re = LogisticWrapper(C=1.0)
forest = RandomForestWrapper(n_estimators=100)
regressions = []
regressions.append(log_re)
regressions.append(forest)
for regression in regressions:
  regression.fit(X_train,y_train)
  y_pred_proba = regression.predict_proba(X_test)
  y_pred = np.where(y_pred_proba > 0.5, 1, 0)
  print(roc_auc_score(y_pred, y_test))
  print(regression)
  print(regression.fit.__name__)

[fit] 0.0037초
[predict_proba] 0.0006초
0.7532051282051283
LogisticWrapper(name='logistic', fitted=True)
fit
[fit] 0.2688초
[predict_proba] 0.0101초
0.7857142857142857
RandomForestWrapper(name='random_forest', fitted=True)
fit


- `_build_model()`을 `@abstractmethod`로 만들지 않으면 무엇이 위험해지는가? 실제로 지우고 하위 클래스에서 구현을 빼먹어보세요.
- `functools.wraps`를 빼면 `LogisticWrapper.fit.__name__`이 무엇으로 나오는가? 직접 확인해보세요.
- 이 구조에 XGBoost 래퍼를 추가하려면 몇 줄을 써야 하는가? 그것이 이 설계의 이점인가?


In [33]:
def log_timing(func):
  """
  요구사항:
  - 감싼 함수의 실행 시간을 측정해 "[함수이름] 0.0123초" 형태로 출력.
  - functools.wraps를 사용해 원래 함수의 __name__과 docstring을 보존할 것.
  """
  #@functools.wraps(func)
  def wrapper(*arg,**kwargs):
    start = time.time()
    result = func(*arg,**kwargs)
    end = time.time()
    print(f"[{func.__name__}] {end-start:.4f}초")
    return result
  return wrapper

class BaseClassifierWrapper(ABC):
  """
  요구사항:
  - __init__(name): 이름과 fitted 상태를 저장.
  - _build_model(): @abstractmethod. 하위 클래스가 실제 sklearn 모델을 반환.
  - fit(X, y): @log_timing 적용. _build_model()로 모델을 만들고 학습, self 반환.
  - predict_proba(X): @log_timing 적용. fit되지 않았으면 명확한 에러.
                      양성 클래스 확률 1차원 배열 반환.
  - __repr__: 클래스명, name, fitted 상태가 보이게.
  """
  def __init__(self,name):
    self.name = name
    self.fitted = False
    self.model = None
  @abstractmethod
  def _build_model(self):
    pass
  @log_timing
  def fit(self, X, y):
    self.model = self._build_model()
    self.model.fit(X,y)
    self.fitted = True
    return self

  @log_timing
  def predict_proba(self, X):
    if not self.fitted:
      raise Exception("모델을 학습해주세요.")
    return self.model.predict_proba(X)[:,1]

  def __repr__(self):
    return f"{self.__class__.__name__}(name='{self.name}', fitted={self.fitted})"

# 위를 상속해서 LogisticWrapper(C=1.0)와 RandomForestWrapper(n_estimators=100)를
# 각각 구현할 것. 두 클래스 모두 _build_model()만 다르게 채우면 되도록 설계.
class LogisticWrapper(BaseClassifierWrapper):
  def __init__(self,C=1.0):
    super().__init__("logistic")
    self.C = C
  def _build_model(self):
    from sklearn.linear_model import LogisticRegression
    return LogisticRegression(C=self.C)

class RandomForestWrapper(BaseClassifierWrapper):
  def __init__(self,n_estimators=100):
    super().__init__("random_forest")
    self.n_estimators = n_estimators
  def _build_model(self):
    from sklearn.ensemble import RandomForestClassifier
    return RandomForestClassifier(n_estimators=self.n_estimators)

In [34]:
X, y = make_classification()
X_train, X_test, y_train, y_test = train_test_split(X,y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
log_re = LogisticWrapper(C=1.0)
forest = RandomForestWrapper(n_estimators=100)
regressions = []
regressions.append(log_re)
regressions.append(forest)
for regression in regressions:
  regression.fit(X_train,y_train)
  y_pred_proba = regression.predict_proba(X_test)
  y_pred = np.where(y_pred_proba > 0.5, 1, 0)
  print(roc_auc_score(y_pred, y_test))
  print(regression)
  print(regression.fit.__name__)

[fit] 0.0414초
[predict_proba] 0.0006초
0.9615384615384616
LogisticWrapper(name='logistic', fitted=True)
wrapper
[fit] 0.4989초
[predict_proba] 0.0380초
0.9615384615384616
RandomForestWrapper(name='random_forest', fitted=True)
wrapper
